# 070 CASE 030 — Marine Vessels: Sotenäs (Bohuslän)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_030-Marine-Vessels-Sotenas.ipynb)

**Purpose:** Automatic vessel detection in Sentinel-2 imagery along the Swedish west coast. Applications: maritime surveillance, illegal fishing, marine pollution, defence intelligence.

**Partners:**
- **Swedish Space Corporation (SSC)** — ground stations, operational experience
- **Sjöfartsverket** — sjökort (S-57)
- **PandionAI** — onboard AI (AlertSat)
- **AI Sweden / RISE** — model development

**Data:** Copernicus Sentinel-2 L2A via DES, fine-tuned YOLO11s (via SAHI for small-object detection), AI2 vessel detector, variance heatmap.

**Licence:** CC0 1.0 notebook. YOLO11s is **AGPL-3.0** — commercial closed-source use requires Ultralytics Enterprise.

## 1. Method

| Analyzer | Purpose | Outputs |
|----------|---------|---------|
| `object_detection` (heatmap) | Variance anomaly map, no model | `regions`, `heatmap` |
| `marine_vessels` | YOLO11s fine-tuned + SAHI slicing | `regions` (bbox dicts), `vessel_count`, `vessel_density_per_km2` |
| `ai2_vessels` | AI2 vessel detector | `regions`, `vessel_count` |
| `spectral` | NDWI water/land discrimination | `indices` dict |

Each `region` is `{bbox: {x_min, y_min, x_max, y_max}, confidence, class_name, ...}` in image pixels.

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
from imint.engine import run_job
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium

def get_outputs(result, name):
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None

AOI = {"west": 11.30, "south": 58.40, "east": 11.55, "north": 58.55}
DATE = "2024-07-15"

print(f"AOI (Sotenäs): {AOI}")
print(f"Date: {DATE}")

## 3. Run analysis

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/marin_sotenas",
    config_path="config/analyzers.yaml",
)

job = executor.build_job(date=DATE, coords=AOI)
result = run_job(job)

mv = get_outputs(result, "marine_vessels")
if mv is not None:
    print(f"YOLO11s: {mv['vessel_count']} vessels")
    print(f"Density: {mv['vessel_density_per_km2']} per km²")
else:
    print("marine_vessels analyzer unavailable (check SAHI + weights).")

ai2 = get_outputs(result, "ai2_vessels")
if ai2 is not None:
    print(f"AI2: {ai2.get('vessel_count', 'n/a')} vessels")

## 4. Visualize

In [ ]:
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite",
).add_to(m)
folium.Rectangle(
    bounds=[[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]],
    color="#0066ff", weight=2, fill=False,
).add_to(m)
folium.LayerControl().add_to(m)
m

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(job.rgb)
axes[0].set_title(f"Sentinel-2 RGB + YOLO detections ({DATE})")
axes[0].axis("off")

if mv is not None:
    for r in mv.get("regions", []):
        bb = r["bbox"]
        rect = mpatches.Rectangle(
            (bb["x_min"], bb["y_min"]),
            bb["x_max"] - bb["x_min"],
            bb["y_max"] - bb["y_min"],
            linewidth=1.5, edgecolor="red", facecolor="none",
        )
        axes[0].add_patch(rect)

od = get_outputs(result, "object_detection")
if od is not None and "heatmap" in od:
    axes[1].imshow(od["heatmap"], cmap="hot")
    axes[1].set_title("Variance heatmap (anomalies)")
else:
    axes[1].text(0.5, 0.5, "object_detection unavailable", ha="center", va="center")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Interpretation

**Sentinel-2 limits for vessel detection:**
- 10 m pixels → vessels ≥ ~15 m detectable
- 5-day revisit → misses fast events
- Cloud-free required → Sentinel-1 SAR complements for all-weather coverage

**PandionAI value:**
- Onboard AI on AlertSat → realtime detection without ground-station latency
- VHR sensors (sub-meter) → smaller vessels detectable

**Use cases:**
- Oil/bunker spill monitoring
- AIS complement (AIS can be switched off — satellite still sees)
- Coast Guard / defence ISR
- Illegal fishing enforcement

### Links
- [ImintEngine/imint/analyzers/marine_vessels.py](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/marine_vessels.py)
- [AI2 Satellite Vessel Detection](https://allenai.org)
- [PandionAI AlertSat](https://www.pandionai.com)